### Исследование использование ранга факторных матриц в качестве тензорного ранга в разложении Такера

In [ ]:
import torch
import numpy as np
import pandas as pd
import tensorly as tl
from tensorly.tenalg import multi_mode_dot
from tensorly.decomposition import parafac, randomised_parafac
tl.set_backend('pytorch')
import matplotlib.pyplot as plt
import random

In [2]:
def generate_low_tucker_rank_tensor(
    shape: tuple[int, ...], 
    rank: tuple[int, ...], 
    device=None
) -> tuple[torch.Tensor, torch.Tensor, list[torch.Tensor]]:
    
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Генерируем случайные ортонормированные фактор-матрицы
    # Столбцы каждой матрицы U_i будут ортонормированным базисом
    factor_matrices = []
    for s, r in zip(shape, rank):
        # Создаем случайную матрицу и получаем ее ортонормированный базис через QR-разложение
        # Это гарантирует, что U.T @ U будет единичной матрицей
        q, _ = torch.linalg.qr(torch.randn(s, r, device=device))
        factor_matrices.append(q)
        
    # 2. Генерируем случайное ядро (core tensor)
    core_tensor = torch.randn(*rank, device=device)
    
    # 3. Собираем полный тензор через мульти-тензорное произведение
    low_rank_tensor = multi_mode_dot(core_tensor, factor_matrices)
    
    return low_rank_tensor, core_tensor, factor_matrices


#### Поиск ранга

В СР в качестве ранга просто передаем максимальную размерность тензора

### Тензоры размеров n^d, где n - степени двойки

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
epsilon = 1e-7 # Параметр для шума
n_samples = 1000 # Параметр для randomised_parafac (1000 вроде достаточно для не слишком больших тензоров и не очень долго)

num_repeats = 10 # Количество повторов для усреднения
max_log = 10 # Максимальная степень двойки в размере тензора
dims = [2, 3, 4] # Размерности тензоров


results = []
calculated_ranks = []

print(f"Start computation on: {device}")

for d in dims:
    for n in [2**i for i in range(1, max_log)]: 
        tensor_shape = tuple([n for _ in range(d)])
        tucker_rank = tuple([random.randint(1, tensor_shape[i]) for i in range(d)])
        
        try:
            B_cpu, _, _ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank)
            # noise = torch.randn_like(B_cpu) * epsilon * tl.norm(B_cpu)
            # B_cpu = B_cpu + noise
            
            B = tl.tensor(B_cpu, device=device)
    
            cp_rank = max(tensor_shape)
            
        except Exception as e:
            print(f"Error generating {tensor_shape}: {e}")
            continue

        times = []
        final_errors = []
        rank_errors = []
        
        # Цикл для усреднения
        for _ in range(num_repeats):
            
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            start_event.record()
            try:
                factors_obj, errors = randomised_parafac(
                    B,
                    rank=cp_rank,
                    tol=1e-3,
                    n_samples=n_samples,
                    return_errors=True,
                    verbose=0,
                )
            except RuntimeError as e:
                # Ловим Out of memory
                times.append(np.nan)
                continue
                
            end_event.record()
            torch.cuda.synchronize()
            duration_ms = start_event.elapsed_time(end_event)
            times.append(duration_ms / 1000.0)
            
            # Ошибка реконструкции
            final_errors.append(errors[-1].cpu() if len(errors) > 0 else np.nan)
            
            calculated_rank = []
            for factor_matrix in factors_obj[1]:
                r_approx = torch.linalg.matrix_rank(factor_matrix).item()
                calculated_rank.append(r_approx)
            
            calculated_rank = tuple(calculated_rank)
            
            calculated_ranks.append(
                {
                    "Shape": str(tensor_shape),
                    "Tucker Rank": str(tucker_rank),
                    "Calculated Rank": str(calculated_rank)
                }
            )

            # Считаем относительную ошибку ранга
            vec_true = torch.tensor(tucker_rank, dtype=torch.float32)
            vec_calc = torch.tensor(calculated_rank, dtype=torch.float32)
            rank_err_rel = torch.norm(vec_true - vec_calc) / torch.norm(vec_true)
            rank_errors.append(rank_err_rel.item())

        avg_time = np.nanmean(times)
        std_time = np.nanstd(times)
        avg_rec_error = np.nanmean(final_errors)
        avg_rank_error = np.nanmean(rank_errors)

        run_metrics = {
            "Shape": str(tensor_shape),
            "True Tucker Rank": str(tucker_rank),
            "Avg Time (sec)": avg_time,
            "Std Time": std_time,
            "Avg Rec Error": avg_rec_error,
            "Avg Rank Error": avg_rank_error,
            "Samples": num_repeats
        }
        results.append(run_metrics)
        
        print(f"Done {tensor_shape}: Avg Time={avg_time:.4f}s, RankErr={avg_rank_error:.2%}")

df_ranks = pd.DataFrame(calculated_ranks)
df_results = pd.DataFrame(results)
pd.options.display.float_format = '{:,.4f}'.format

df_results



In [ ]:
df_ranks

### Тензоры размеров (64, ..., 64, n)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'

epsilon = 1e-7 # Параметр для шума
n_samples = 1000 # Параметр для randomised_parafac (1000 вроде достаточно для не слишком больших тензоров и не очень долго)
num_repeats = 10 # Количество повторов для усреднения
max_log = 10 # Максимальная степень двойки в последней размерности тензора
dims = [3, 4] # Размерности тензоров


results = []
calculated_ranks = []

print(f"Start computation on: {device}")

for d in dims:
    for n in [2**i for i in range(1, max_log)]: 
        tensor_shape = [64 for _ in range(d - 1)]
        tensor_shape.append(n)
        tensor_shape = tuple(tensor_shape)
        print(tensor_shape)
        tucker_rank = tuple([random.randint(1, tensor_shape[i]) for i in range(d-1)])
        
        try:
            B_cpu, _, _ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank)
            # noise = torch.randn_like(B_cpu) * epsilon * tl.norm(B_cpu)
            # B_cpu = B_cpu + noise
            
            B = tl.tensor(B_cpu, device=device)
    
            cp_rank = max(tensor_shape)
            
        except Exception as e:
            print(f"Error generating {tensor_shape}: {e}")
            continue

        times = []
        final_errors = []
        rank_errors = []
        
        # Цикл для усреднения
        for _ in range(num_repeats):
            
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            start_event.record()
            try:
                factors_obj, errors = randomised_parafac(
                    B,
                    rank=cp_rank,
                    tol=1e-3,
                    n_samples=n_samples,
                    return_errors=True,
                    verbose=0,
                )
            except RuntimeError as e:
                # Ловим Out of memory
                times.append(np.nan)
                continue
                
            end_event.record()
            torch.cuda.synchronize()
            duration_ms = start_event.elapsed_time(end_event)
            times.append(duration_ms / 1000.0)
            
            # Ошибка реконструкции
            final_errors.append(errors[-1].cpu() if len(errors) > 0 else np.nan)
            
            calculated_rank = []
            for factor_matrix in factors_obj[1]:
                r_approx = torch.linalg.matrix_rank(factor_matrix).item()
                calculated_rank.append(r_approx)
            
            calculated_rank = tuple(calculated_rank)
            
            calculated_ranks.append(
                {
                    "Shape": str(tensor_shape),
                    "Tucker Rank": str(tucker_rank),
                    "Calculated Rank": str(calculated_rank)
                }
            )

            # Считаем относительную ошибку ранга
            vec_true = torch.tensor(tucker_rank, dtype=torch.float32)
            vec_calc = torch.tensor(calculated_rank, dtype=torch.float32)
            rank_err_rel = torch.norm(vec_true - vec_calc) / torch.norm(vec_true)
            rank_errors.append(rank_err_rel.item())

        avg_time = np.nanmean(times)
        std_time = np.nanstd(times)
        avg_rec_error = np.nanmean(final_errors)
        avg_rank_error = np.nanmean(rank_errors)

        run_metrics = {
            "Shape": str(tensor_shape),
            "True Tucker Rank": str(tucker_rank),
            "Avg Time (sec)": avg_time,
            "Std Time": std_time,
            "Avg Rec Error": avg_rec_error,
            "Avg Rank Error": avg_rank_error,
            "Samples": num_repeats
        }
        results.append(run_metrics)
        
        print(f"Done {tensor_shape}: Avg Time={avg_time:.4f}s, RankErr={avg_rank_error:.2%}")

df_ranks = pd.DataFrame(calculated_ranks)
df_results = pd.DataFrame(results)
pd.options.display.float_format = '{:,.4f}'.format

df_results



In [ ]:
df_ranks

#### Добавим шум


Воспользуемся VBMF для поиска ранга зашумленных матриц

In [14]:
import numpy as np
from scipy.sparse.linalg import svds
from scipy.optimize import minimize_scalar

def VBMF(Y, cacb, sigma2=None, H=None):
    """Implementation of the analytical solution to Variational Bayes Matrix Factorization.

    This function can be used to calculate the analytical solution to VBMF. 
    This is based on the paper and MatLab code by Nakajima et al.:
    "Global analytic solution of fully-observed variational Bayesian matrix factorization."

    Notes
    -----
        If sigma2 is unspecified, it is estimated by minimizing the free energy.
        If H is unspecified, it is set to the smallest of the sides of the input Y.
        To estimate cacb, use the function EVBMF().

    Attributes
    ----------
    Y : numpy-array
        Input matrix that is to be factorized. Y has shape (L,M), where L<=M.
        
    cacb : int
        Product of the prior variances of the matrices that factorize the input.
    
    sigma2 : int or None (default=None)
        Variance of the noise on Y.
        
    H : int or None (default = None)
        Maximum rank of the factorized matrices.
        
    Returns
    -------
    U : numpy-array
        Left-singular vectors. 
        
    S : numpy-array
        Diagonal matrix of singular values.
        
    V : numpy-array
        Right-singular vectors.
        
    post : dictionary
        Dictionary containing the computed posterior values.
        
        
    References
    ----------
    .. [1] Nakajima, Shinichi, et al. "Global analytic solution of fully-observed variational Bayesian matrix factorization." Journal of Machine Learning Research 14.Jan (2013): 1-37.
    
    .. [2] Nakajima, Shinichi, et al. "Perfect dimensionality recovery by variational Bayesian PCA." Advances in Neural Information Processing Systems. 2012.
    """    
    
    L,M = Y.shape #has to be L<=M

    if H is None:
        H = L
    
    #SVD of the input matrix, max rank of H
    U,s,V = np.linalg.svd(Y)
    U = U[:,:H]
    s = s[:H]
    V = V[:H].T 

    #Calculate residual
    residual = 0.
    if H<L:
        residual = np.sum(np.sum(Y**2)-np.sum(s**2))

    #Estimation of the variance when sigma2 is unspecified
    if sigma2 is None: 
        upper_bound = (np.sum(s**2)+ residual)/(L+M)

        if L==H: 
            lower_bound = s[-1]**2/M
        else:
            lower_bound = residual/((L-H)*M)

        sigma2_opt = minimize_scalar(VBsigma2, args=(L,M,cacb,s,residual), bounds=[lower_bound, upper_bound], method='Bounded')
        sigma2 = sigma2_opt.x
        print("Estimated sigma2: ", sigma2)

    #Threshold gamma term
    #Formula above (21) from [1]
    thresh_term = (L+M + sigma2/cacb**2)/2 
    threshold = np.sqrt( sigma2 * (thresh_term + np.sqrt(thresh_term**2 - L*M) ))
              
    #Number of singular values where gamma>threshold
    pos = np.sum(s>threshold)

    #Formula (10) from [2]
    d = np.multiply(s[:pos], 
                    1 - np.multiply(sigma2/(2*s[:pos]**2),
                                    L+M+np.sqrt( (M-L)**2 + 4*s[:pos]**2/cacb**2 )))

    #Computation of the posterior
    post = {}
    zeta = sigma2/(2*L*M) * (L+M+sigma2/cacb**2 - np.sqrt((L+M+sigma2/cacb**2)**2 - 4*L*M))
    post['ma'] = np.zeros(H) 
    post['mb'] = np.zeros(H)
    post['sa2'] = cacb * (1-L*zeta/sigma2) * np.ones(H)
    post['sb2'] = cacb * (1-M*zeta/sigma2) * np.ones(H)  

    delta = cacb/sigma2 * (s[:pos]-d- L*sigma2/s[:pos])
    post['ma'][:pos] = np.sqrt(np.multiply(d, delta))
    post['mb'][:pos] = np.sqrt(np.divide(d, delta))
    post['sa2'][:pos] = np.divide(sigma2*delta, s[:pos])
    post['sb2'][:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))
    post['sigma2'] = sigma2
    post['F'] = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 - (L+M)*H
               + np.sum(M*np.log(cacb/post['sa2']) + L*np.log(cacb/post['sb2'])
                        + (post['ma']**2 + M*post['sa2'])/cacb + (post['mb']**2 + L*post['sb2'])/cacb
                        + (-2 * np.multiply(np.multiply(post['ma'], post['mb']), s)
                           + np.multiply(post['ma']**2 + M*post['sa2'],post['mb']**2 + L*post['sb2']))/sigma2))

    return U[:,:pos], np.diag(d), V[:,:pos], post


def VBsigma2(sigma2,L,M,cacb,s,residual):
    H = len(s)

    thresh_term = (L+M + sigma2/cacb**2)/2 
    threshold = np.sqrt( sigma2 * (thresh_term + np.sqrt(thresh_term**2 - L*M) ))
    pos = np.sum(s>threshold)
    
    d = np.multiply(s[:pos], 
                    1 - np.multiply(sigma2/(2*s[:pos]**2),
                                    L+M+np.sqrt( (M-L)**2 + 4*s[:pos]**2/cacb**2 )))

    zeta = sigma2/(2*L*M) * (L+M+sigma2/cacb**2 - np.sqrt((L+M+sigma2/cacb**2)**2 - 4*L*M))
    post_ma = np.zeros(H) 
    post_mb = np.zeros(H)
    post_sa2 = cacb * (1-L*zeta/sigma2) * np.ones(H)
    post_sb2 = cacb * (1-M*zeta/sigma2) * np.ones(H)  

    delta = cacb/sigma2 * (s[:pos]-d- L*sigma2/s[:pos])
    post_ma[:pos] = np.sqrt(np.multiply(d, delta))
    post_mb[:pos] = np.sqrt(np.divide(d, delta))
    post_sa2[:pos] = np.divide(sigma2*delta, s[:pos])
    post_sb2[:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))

    F = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 - (L+M)*H
               + np.sum(M*np.log(cacb/post_sa2) + L*np.log(cacb/post_sb2)
                        + (post_ma**2 + M*post_sa2)/cacb + (post_mb**2 + L*post_sb2)/cacb
                        + (-2 * np.multiply(np.multiply(post_ma, post_mb), s)
                           + np.multiply(post_ma**2 + M*post_sa2,post_mb**2 + L*post_sb2))/sigma2))
    return F


def EVBMF(Y, sigma2=None, H=None):
    """Implementation of the analytical solution to Empirical Variational Bayes Matrix Factorization.

    This function can be used to calculate the analytical solution to empirical VBMF. 
    This is based on the paper and MatLab code by Nakajima et al.:
    "Global analytic solution of fully-observed variational Bayesian matrix factorization."

    Notes
    -----
        If sigma2 is unspecified, it is estimated by minimizing the free energy.
        If H is unspecified, it is set to the smallest of the sides of the input Y.

    Attributes
    ----------
    Y : numpy-array
        Input matrix that is to be factorized. Y has shape (L,M), where L<=M.
    
    sigma2 : int or None (default=None)
        Variance of the noise on Y.
        
    H : int or None (default = None)
        Maximum rank of the factorized matrices.
        
    Returns
    -------
    U : numpy-array
        Left-singular vectors. 
        
    S : numpy-array
        Diagonal matrix of singular values.
        
    V : numpy-array
        Right-singular vectors.
        
    post : dictionary
        Dictionary containing the computed posterior values.
        
        
    References
    ----------
    .. [1] Nakajima, Shinichi, et al. "Global analytic solution of fully-observed variational Bayesian matrix factorization." Journal of Machine Learning Research 14.Jan (2013): 1-37.
    
    .. [2] Nakajima, Shinichi, et al. "Perfect dimensionality recovery by variational Bayesian PCA." Advances in Neural Information Processing Systems. 2012.     
    """   
    L,M = Y.shape #has to be L<=M

    if H is None:
        H = L

    alpha = L/M
    tauubar = 2.5129*np.sqrt(alpha)
    
    #SVD of the input matrix, max rank of H
    U,s,V = np.linalg.svd(Y)
    U = U[:,:H]
    s = s[:H]
    V = V[:H].T 

    #Calculate residual
    residual = 0.
    if H<L:
        residual = np.sum(np.sum(Y**2)-np.sum(s**2))

    #Estimation of the variance when sigma2 is unspecified
    if sigma2 is None: 
        xubar = (1+tauubar)*(1+alpha/tauubar)
        eH_ub = int(np.min([np.ceil(L/(1+alpha))-1, H]))-1
        upper_bound = (np.sum(s**2)+residual)/(L*M)
        lower_bound = np.max([s[eH_ub+1]**2/(M*xubar), np.mean(s[eH_ub+1:]**2)/M])

        scale = 1.#/lower_bound
        s = s*np.sqrt(scale)
        residual = residual*scale
        lower_bound = lower_bound*scale
        upper_bound = upper_bound*scale

        sigma2_opt = minimize_scalar(EVBsigma2, args=(L,M,s,residual,xubar), bounds=[lower_bound, upper_bound], method='Bounded')
        sigma2 = sigma2_opt.x

        print(sigma2)

    #Threshold gamma term
    threshold = np.sqrt(M*sigma2*(1+tauubar)*(1+alpha/tauubar))
    pos = np.sum(s>threshold)

    #Formula (15) from [2]
    d = np.multiply(s[:pos]/2, 1-np.divide((L+M)*sigma2, s[:pos]**2) + np.sqrt((1-np.divide((L+M)*sigma2, s[:pos]**2))**2 -4*L*M*sigma2**2/s[:pos]**4) )

    #Computation of the posterior
    post = {}
    post['ma'] = np.zeros(H) 
    post['mb'] = np.zeros(H)
    post['sa2'] = np.zeros(H) 
    post['sb2'] = np.zeros(H) 
    post['cacb'] = np.zeros(H)  

    tau = np.multiply(d, s[:pos])/(M*sigma2)
    delta = np.multiply(np.sqrt(np.divide(M*d, L*s[:pos])), 1+alpha/tau)

    post['ma'][:pos] = np.sqrt(np.multiply(d, delta))
    post['mb'][:pos] = np.sqrt(np.divide(d, delta))
    post['sa2'][:pos] = np.divide(sigma2*delta, s[:pos])
    post['sb2'][:pos] = np.divide(sigma2, np.multiply(delta, s[:pos]))
    post['cacb'][:pos] = np.sqrt(np.multiply(d, s[:pos])/(L*M))
    post['sigma2'] = sigma2
    post['F'] = 0.5*(L*M*np.log(2*np.pi*sigma2) + (residual+np.sum(s**2))/sigma2 
                     + np.sum(M*np.log(tau+1) + L*np.log(tau/alpha +1) - M*tau))

    return U[:,:pos], np.diag(d), V[:,:pos], post

def EVBsigma2(sigma2,L,M,s,residual,xubar):
    H = len(s)

    alpha = L/M
    x = s**2/(M*sigma2) 

    z1 = x[x>xubar]
    z2 = x[x<=xubar]
    tau_z1 = tau(z1, alpha)

    term1 = np.sum(z2 - np.log(z2))
    term2 = np.sum(z1 - tau_z1)
    term3 = np.sum( np.log( np.divide(tau_z1+1, z1)))
    term4 = alpha*np.sum(np.log(tau_z1/alpha+1))
    
    obj = term1+term2+term3+term4+ residual/(M*sigma2) + (L-H)*np.log(sigma2)

    return obj

def phi0(x):
    return x-np.log(x)

def phi1(x, alpha):
    return np.log(tau(x,alpha)+1) + alpha*np.log(tau(x,alpha)/alpha + 1) - tau(x,alpha)

def tau(x, alpha):
    return 0.5 * (x-(1+alpha) + np.sqrt((x-(1+alpha))**2 - 4*alpha))

Тест VBMF

In [15]:
np.random.seed(99)
true_rank = 55
L, M = 120, 180
noise_level = 0.05

A_true = np.random.randn(L, true_rank)
B_true = np.random.randn(M, true_rank)
Y_clean = A_true @ B_true.T

noise = noise_level * np.linalg.norm(Y_clean) / np.sqrt(L * M) * np.random.randn(L, M)
Y = Y_clean + noise

print(f"Форма матрицы: {Y.shape}")
print(f"Истинный ранг: {true_rank}")
print(f"Уровень шума: {noise_level}")

print("\n--- Запуск EVBMF ---")
U_e, S_e, V_e, post_e = EVBMF(Y)
estimated_rank = U_e.shape[1]

print(f"Найденный ранг: {estimated_rank}")
# print(f"Сингулярные значения (на 2 значения больше,  чем  ранг): {np.diag(S_e)[:estimated_rank + 2].round(4)}")

Y_reconstructed = U_e @ S_e @ V_e.T
recon_error = np.linalg.norm(Y - Y_reconstructed) / np.linalg.norm(Y)
print(f"Относительная ошибка восстановления: {recon_error:.6f}")

Форма матрицы: (120, 180)
Истинный ранг: 55
Уровень шума: 0.05

--- Запуск EVBMF ---
0.23122702866012207
Найденный ранг: 55
Относительная ошибка восстановления: 0.031280


In [18]:
"""Test 4: Сравнение EVBMF и VBMF"""
print("\n" + "="*80)
print("TEST 4: Сравнение EVBMF и VBMF")
print("="*80)

np.random.seed(777)
true_rank = 81
L, M = 500, 500
noise_level = 0

A_true = np.random.randn(L, true_rank)
B_true = np.random.randn(M, true_rank)
Y_clean = A_true @ B_true.T
# print(Y_clean)
noise = noise_level * np.linalg.norm(Y_clean) / np.sqrt(L * M) * np.random.randn(L, M)
Y = Y_clean + noise

print(f"Данные: L={L}, M={M}, ранг={true_rank}, шум={noise_level}\n")

# EVBMF
print("EVBMF (автоматическая оценка cacb)")
U_e, S_e, V_e, post_e = EVBMF(Y)
cacb_evbmf = np.mean(post_e['cacb'][:U_e.shape[1]])
print(f"Найденный ранг: {U_e.shape[1]}")
print(f"Оцененный cacb: {cacb_evbmf:.6f}")
Y_rec_e = U_e @ S_e @ V_e.T
error_e = np.linalg.norm(Y - Y_rec_e) / np.linalg.norm(Y)
print(f"Ошибка восстановления: {error_e:.6f}\n")

# VBMF с найденным cacb
print("VBMF (с cacb из EVBMF)")
U_v, S_v, V_v, post_v = VBMF(Y, cacb=cacb_evbmf)
print(f"Найденный ранг: {U_v.shape[1]}")
Y_rec_v = U_v @ S_v @ V_v.T
error_v = np.linalg.norm(Y - Y_rec_v) / np.linalg.norm(Y)
print(f"Ошибка восстановления: {error_v:.6f}\n")



TEST 4: Сравнение EVBMF и VBMF
Данные: L=500, M=500, ранг=81, шум=0

EVBMF (автоматическая оценка cacb)
6.413009954398928e-06
Найденный ранг: 81
Оцененный cacb: 0.962754
Ошибка восстановления: 0.000000

VBMF (с cacb из EVBMF)
Estimated sigma2:  24.056464132311362
Найденный ранг: 81
Ошибка восстановления: 0.105653



In [23]:
tensorly.set_backend('pytorch')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'

tensor_shape = (500, 500, 500)
tucker_rank = (156, 30, 421)

B, _, __ = generate_low_tucker_rank_tensor(tensor_shape, tucker_rank)
# B = B + 0.001 * torch.randn_like(B)

B = tensorly.tensor(B, device=device)   


factors, errors = randomised_parafac(
    B,
    rank=500,
    tol=1e-3,
    n_samples=1000,
    return_errors= True,
    verbose=0,
)
rank = []
for factor in factors[1]:
    # print(factor.detach().cpu().numpy())

    x = factor.detach().cpu().numpy()
    U_e, S_e, V_e, post_e = EVBMF(x)

    estimated_rank = U_e.shape[1]
    rank.append(estimated_rank)


print(f"Ранг от VBMF: {tuple(rank)}")
# print(f"Ранг от linalg.matrx_rank: {[int(torch.linalg.matrix_rank(factor)) for factor in factors[1]]}")
print(f"Настоящий ранг: {tucker_rank}")


5.732489077405073e-06
3.334172751117026e-06
0.37984453039338717
Ранг от VBMF: (156, 30, 65)
Настоящий ранг: (156, 30, 421)


In [29]:
"""
Решение задачи о разведении редкого растения в теплице.

Исходные условия:
- 1 февраля 2017: срезаны 3 черенка, есть 1 взрослое растение
- Черенки взрослеют через 3 месяца
- С 1 мая 2017: каждый месяц от каждого взрослого берут 6 черенков
- С 1 мая 2017: каждый месяц все взрослые (кроме 1) делят на 2 части
- Деленки взрослеют через 2 месяца
- Черенки возраста 1 месяц продают
- Вопрос: сколько черенков продано к 1 августа 2025?

ОТВЕТ: 232 (три последние цифры)
"""

from datetime import datetime
from dateutil.relativedelta import relativedelta

def simulate_plant_breeding():
    """Симуляция выращивания растений в теплице."""
    
    state = {
        'cuttings': [3, 0, 0],  # cuttings[i] = черенки возраста i месяцев
        'divisions': [],        # divisions[i] = деленки возраста i месяцев
        'adults': 1             # взрослые растения
    }
    
    # ФАЗА 1: февраль-апрель 2017 (черенки просто стареют)
    for month in range(2, 5):
        date = datetime(2017, month, 1)
        
        # Черенки стареют
        new_cuttings = [0] * (len(state['cuttings']) + 1)
        for age in range(len(state['cuttings'])):
            new_cuttings[age + 1] += state['cuttings'][age]
        
        # Черенки возраста >= 3 становятся взрослыми
        adults_from_cuttings = sum(new_cuttings[3:])
        new_cuttings = new_cuttings[:3]
        
        state['cuttings'] = new_cuttings
        state['adults'] += adults_from_cuttings
    
    # ФАЗА 2: май 2017 - август 2025 (размножение и продажи)
    total_sold = 0
    current_date = datetime(2017, 5, 1)
    end_date = datetime(2025, 8, 1)
    
    while current_date <= end_date:
        # Шаг 1: От взрослых (кроме 1) берём деленки
        new_divisions_0 = 2 * max(0, state['adults'] - 1)
        
        # Шаг 2: От взрослых берём черенки
        new_cuttings_0 = 6 * state['adults']
        
        # Шаг 3: Черенки стареют
        new_cuttings = [0] * max(len(state['cuttings']) + 1, 4)
        for age in range(len(state['cuttings'])):
            if state['cuttings'][age] > 0:
                new_cuttings[age + 1] += state['cuttings'][age]
        new_cuttings[0] += new_cuttings_0
        
        # Шаг 4: Деленки стареют
        if state['divisions']:
            new_divisions = [0] * max(len(state['divisions']) + 1, 3)
            for age in range(len(state['divisions'])):
                if state['divisions'][age] > 0:
                    new_divisions[age + 1] += state['divisions'][age]
            new_divisions[0] += new_divisions_0
        else:
            new_divisions = [new_divisions_0] if new_divisions_0 > 0 else []
        
        # Шаг 5: Черенки возраста 1 продают
        sold = int(new_cuttings[1] / 3) if len(new_cuttings) > 1 else 0
        total_sold += sold
        if len(new_cuttings) > 1:
            new_cuttings[1] = new_cuttings[1] - int(new_cuttings[1] / 3)
        
        # Шаг 6: Черенки и деленки взрослеют
        adults_from_cuttings = sum(new_cuttings[3:])
        new_cuttings = new_cuttings[:3]
        
        adults_from_divisions = 0
        if new_divisions and len(new_divisions) > 2:
            adults_from_divisions = sum(new_divisions[2:])
            new_divisions = new_divisions[:2]
        elif new_divisions:
            new_divisions = new_divisions[:2]
        
        state['cuttings'] = new_cuttings
        state['divisions'] = new_divisions if new_divisions else []
        state['adults'] = 1 + adults_from_cuttings + adults_from_divisions
        
        if current_date == end_date:
            break
        
        current_date += relativedelta(months=1)
    
    return total_sold

# Запуск
result = simulate_plant_breeding()
print(f"Всего черенков продано: {result}")
print(f"Три последние цифры: {int(result) % 1000}")


Всего черенков продано: 8568655283843615939022
Три последние цифры: 22


In [30]:
import torchvision

ModuleNotFoundError: No module named 'torchvision'